In [3]:
import os, sys
sys.path.append(os.path.abspath(os.path.join("..")))

In [4]:
from src.data import load_csv

In [5]:
df = load_csv("../data/hour.csv")

- instant: record index
	- dteday : date
	- season : season (1:springer, 2:summer, 3:fall, 4:winter)
	- yr : year (0: 2011, 1:2012)
	- mnth : month ( 1 to 12)
	- hr : hour (0 to 23)
	- holiday : weather day is holiday or not (extracted from http://dchr.dc.gov/page/holiday-schedule)
	- weekday : day of the week
	- workingday : if day is neither weekend nor holiday is 1, otherwise is 0.
	+ weathersit : 
		- 1: Clear, Few clouds, Partly cloudy, Partly cloudy
		- 2: Mist + Cloudy, Mist + Broken clouds, Mist + Few clouds, Mist
		- 3: Light Snow, Light Rain + Thunderstorm + Scattered clouds, Light Rain + Scattered clouds
		- 4: Heavy Rain + Ice Pallets + Thunderstorm + Mist, Snow + Fog
	- temp : Normalized temperature in Celsius. The values are divided to 41 (max)
	- atemp: Normalized feeling temperature in Celsius. The values are divided to 50 (max)
	- hum: Normalized humidity. The values are divided to 100 (max)
	- windspeed: Normalized wind speed. The values are divided to 67 (max)
	- casual: count of casual users
	- registered: count of registered users
	- cnt: count of total rental bikes including both casual and registered

we have already neutralised `index`, `casual`, `registered` while loading the dataset, already transformed the `dteday` column from object to datetime. Now let us further transform our dataset after extracting what we need.

Yesterday, I log-transformed target column: `cnt` and after training our dataset only on `atemp, hum and windspeed` our test R2 score went up from 9.6 to 15.07,  RMSE dropped from 209 to 151. This is why data transformation is needed for our model to understand real hidden patterns.

Our dear ledger checkpoints:


In [6]:
df

,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,cnt
0,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0000,16
1,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0000,40
2,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0000,32
3,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0000,13
4,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0000,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17374,2012-12-31,1,1,12,19,0,1,1,2,0.26,0.2576,0.60,0.1642,119
17375,2012-12-31,1,1,12,20,0,1,1,2,0.26,0.2576,0.60,0.1642,89
17376,2012-12-31,1,1,12,21,0,1,1,1,0.26,0.2576,0.60,0.1642,90
17377,2012-12-31,1,1,12,22,0,1,1,1,0.26,0.2727,0.56,0.1343,61


What each column represents as per the official documentation: 

In [11]:
df['weathersit'].value_counts().reset_index()

,weathersit,count
0,1,11413
1,2,4544
2,3,1419
3,4,3


In [13]:
df['hum'] * 100

0        81.0
1        80.0
2        80.0
3        75.0
4        75.0
         ... 
17374    60.0
17375    60.0
17376    60.0
17377    56.0
17378    65.0
Name: hum, Length: 17379, dtype: float64